In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

In [2]:
df = pd.read_csv("../../Datasets_For_Model_Training/Final_Merged_Dataset.csv")
df.head()

,Date,Electricity_Requirement,Humidity,Rainfall,Electricity_Supply,Solar_Irradiance,Temperature
0,01-04-2015,8361.0,70.07,130.566857,8112.0,166.53,28.57
1,01-05-2015,8381.0,77.22,160.792286,8165.0,155.03,27.95
2,01-06-2015,8302.0,77.55,98.240286,8257.0,160.31,27.31
3,01-07-2015,8953.0,73.18,37.804286,8901.0,166.41,27.68
4,01-08-2015,8535.0,72.58,63.944286,8531.0,167.12,27.85


In [3]:
df["Date"]=pd.to_datetime(df["Date"])

In [4]:
# extracting year and month
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month

In [5]:
import numpy as np

# Cyclical Encoding of Month
df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)

# Verify the encoding
print(
    df[["Month", "Month_sin", "Month_cos"]]
    .drop_duplicates()
    .sort_values("Month")
)

   Month  Month_sin  Month_cos
0      1        0.5   0.866025


In [6]:
# Create lag features for Electricity Requirement

df["Demand_Lag_1"] = df["Electricity_Requirement"].shift(1)
df["Demand_Lag_2"] = df["Electricity_Requirement"].shift(2)
df["Demand_Lag_3"] = df["Electricity_Requirement"].shift(3)


# Remove rows with missing lag values
df = df.dropna().reset_index(drop=True)

print(df.head())

        Date  Electricity_Requirement  Humidity    Rainfall  \
0 2015-01-07                   8953.0     73.18   37.804286   
1 2015-01-08                   8535.0     72.58   63.944286   
2 2015-01-09                   8498.0     75.12  119.769714   
3 2015-01-10                   8330.0     78.86  153.807429   
4 2015-01-11                   6511.0     87.29  316.494857   

   Electricity_Supply  Solar_Irradiance  Temperature  Year  Month  Month_sin  \
0              8901.0            166.41        27.68  2015      1        0.5   
1              8531.0            167.12        27.85  2015      1        0.5   
2              8373.0            157.25        27.50  2015      1        0.5   
3              8324.0            143.69        26.60  2015      1        0.5   
4              6508.0             92.62        25.04  2015      1        0.5   

   Month_cos  Demand_Lag_1  Demand_Lag_2  Demand_Lag_3  
0   0.866025        8302.0        8381.0        8361.0  
1   0.866025        8953.0

In [7]:
y = df["Electricity_Requirement"]

In [8]:
X = df[
    [
        "Humidity",
        "Rainfall",
        "Solar_Irradiance",
        "Temperature",
        "Year",
        "Month_sin",
        "Month_cos"
    ]
]

In [9]:
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print(X_train.shape)
print(X_test.shape)

(100, 7)
(26, 7)


In [10]:
print(X_train.index[:5])

print(y_train.index[:5])

print(X_train.index.equals(y_train.index))

RangeIndex(start=0, stop=5, step=1)
RangeIndex(start=0, stop=5, step=1)
True


In [11]:
model = SARIMAX(
    y_train,
    exog=X_train,
    order=(1,1,1),
    seasonal_order=(1,1,1,12),
    enforce_stationarity=False,
    enforce_invertibility=False
)

results = model.fit(
    maxiter=500,
    disp=False
)

In [12]:
print(results.mle_retvals)

{'fopt': 5.498418589209047, 'gopt': array([ 3.02566860e-06,  5.69411185e-07,  8.99014196e-07,  1.77623249e-05,
       -3.83160170e-07, -3.55271368e-10, -7.99360578e-10, -2.56539678e-05,
       -4.01966460e-05, -1.03310249e-05, -3.73885811e-05, -1.52944324e-07]), 'fcalls': 2249, 'warnflag': 0, 'converged': True, 'iterations': 157}


In [13]:
train_pred = results.predict(
    start=y_train.index[0],
    end=y_train.index[-1],
    exog=X_train
)

test_pred = results.predict(
    start=y_test.index[0],
    end=y_test.index[-1],
    exog=X_test
)

In [14]:
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

# ============================
# Predictions
# ============================

train_predictions = results.predict(
    start=y_train.index[0],
    end=y_train.index[-1],
    exog=X_train
)

test_predictions = results.predict(
    start=y_test.index[0],
    end=y_test.index[-1],
    exog=X_test
)

# ============================
# Training Metrics
# ============================

train_mae = mean_absolute_error(y_train, train_predictions)
train_rmse = np.sqrt(mean_squared_error(y_train, train_predictions))
train_mape = mean_absolute_percentage_error(y_train, train_predictions) * 100
train_r2 = r2_score(y_train, train_predictions)

# ============================
# Testing Metrics
# ============================

test_mae = mean_absolute_error(y_test, test_predictions)
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
test_mape = mean_absolute_percentage_error(y_test, test_predictions) * 100
test_r2 = r2_score(y_test, test_predictions)

# ============================
# Print Results
# ============================

print("=" * 50)
print("Training Performance")
print("=" * 50)
print(f"MAE  : {train_mae:.2f}")
print(f"RMSE : {train_rmse:.2f}")
print(f"MAPE : {train_mape:.2f}%")
print(f"R²   : {train_r2:.4f}")

print("\n")

print("=" * 50)
print("Testing Performance")
print("=" * 50)
print(f"MAE  : {test_mae:.2f}")
print(f"RMSE : {test_rmse:.2f}")
print(f"MAPE : {test_mape:.2f}%")
print(f"R²   : {test_r2:.4f}")

Training Performance
MAE  : 323.66
RMSE : 448.20
MAPE : 3.65%
R²   : 0.7715


Testing Performance
MAE  : 454.84
RMSE : 569.52
MAPE : 4.31%
R²   : 0.6645


In [15]:
from sklearn.linear_model import LinearRegression

In [16]:
# Create model
lr_model = LinearRegression()

# Train
lr_model.fit(X_train, y_train)

LinearRegression()

In [17]:
# Training Predictions
train_predictions = lr_model.predict(X_train)

# Testing Predictions
test_predictions = lr_model.predict(X_test)

In [18]:
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

# ==========================
# Training Metrics
# ==========================

train_mae = mean_absolute_error(y_train, train_predictions)
train_rmse = np.sqrt(mean_squared_error(y_train, train_predictions))
train_mape = mean_absolute_percentage_error(y_train, train_predictions) * 100
train_r2 = r2_score(y_train, train_predictions)

# ==========================
# Testing Metrics
# ==========================

test_mae = mean_absolute_error(y_test, test_predictions)
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
test_mape = mean_absolute_percentage_error(y_test, test_predictions) * 100
test_r2 = r2_score(y_test, test_predictions)

print("=" * 50)
print("Training Performance")
print("=" * 50)
print(f"MAE  : {train_mae:.2f}")
print(f"RMSE : {train_rmse:.2f}")
print(f"MAPE : {train_mape:.2f}%")
print(f"R²   : {train_r2:.4f}")

print("\n")

print("=" * 50)
print("Testing Performance")
print("=" * 50)
print(f"MAE  : {test_mae:.2f}")
print(f"RMSE : {test_rmse:.2f}")
print(f"MAPE : {test_mape:.2f}%")
print(f"R²   : {test_r2:.4f}")

Training Performance
MAE  : 389.12
RMSE : 526.10
MAPE : 4.33%
R²   : 0.6851


Testing Performance
MAE  : 583.24
RMSE : 710.95
MAPE : 5.21%
R²   : 0.4772
